# Lab 2 - Data Collection with Web Scraping

Scrape the Wikipedia 'List of Falcon 9 and Falcon Heavy launches' page with BeautifulSoup and parse the launch records table into a dataframe.

In [1]:
import requests, unicodedata
import pandas as pd
from bs4 import BeautifulSoup

In [2]:
static_url = 'https://en.wikipedia.org/w/index.php?title=List_of_Falcon_9_and_Falcon_Heavy_launches&oldid=1027686922'
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36'}
response = requests.get(static_url, headers=headers)
soup = BeautifulSoup(response.text, 'html.parser')
print('page title:', soup.title.string)

page title: List of Falcon 9 and Falcon Heavy launches - Wikipedia


### Helper functions to parse the table cells

In [3]:
def date_time(table_cells):
    return [dt.strip() for dt in list(table_cells.strings)][0:2]

def booster_version(table_cells):
    return ''.join([bv for i,bv in enumerate(table_cells.strings) if i%2==0][0:-1])

def landing_status(table_cells):
    return [i for i in table_cells.strings][0]

def get_mass(table_cells):
    mass = unicodedata.normalize('NFKD', table_cells.text).strip()
    if mass:
        mass = mass[0:mass.find('kg')+2]
    else:
        mass = 0
    return mass

def extract_column_from_header(row):
    if row.br: row.br.extract()
    if row.a: row.a.extract()
    if row.sup: row.sup.extract()
    column_name = ' '.join(row.contents)
    if not column_name.strip().isdigit():
        return column_name.strip()

### Find the launch records table and extract the column names

In [4]:
html_tables = soup.find_all('table')
first_launch_table = html_tables[2]
column_names = []
for th in first_launch_table.find_all('th'):
    name = extract_column_from_header(th)
    if name is not None and len(name) > 0:
        column_names.append(name)
print(column_names[:11])

['Flight No.', 'Date and time ( )', 'Launch site', 'Payload', 'Payload mass', 'Orbit', 'Customer', 'Launch outcome']


### Parse the rows into a dictionary and build the dataframe

In [5]:
launch_dict = {c: [] for c in ['Flight No.','Date','Time','Version Booster','Launch site','Payload','Payload mass','Orbit','Customer','Launch outcome','Booster landing']}
for table in soup.find_all('table', 'wikitable plainrowheaders collapsible'):
    for rows in table.find_all('tr'):
        flag = bool(rows.th and rows.th.string and rows.th.string.strip().isdigit())
        row = rows.find_all('td')
        if flag:
            launch_dict['Flight No.'].append(rows.th.string.strip())
            dt = date_time(row[0]); launch_dict['Date'].append(dt[0].strip(',')); launch_dict['Time'].append(dt[1] if len(dt)>1 else '')
            launch_dict['Version Booster'].append(booster_version(row[1]) or (row[1].a.string if row[1].a else ''))
            launch_dict['Launch site'].append(row[2].a.string if row[2].a else row[2].get_text(strip=True))
            launch_dict['Payload'].append(row[3].a.string if row[3].a else row[3].get_text(strip=True))
            launch_dict['Payload mass'].append(get_mass(row[4]))
            launch_dict['Orbit'].append(row[5].a.string if row[5].a else row[5].get_text(strip=True))
            launch_dict['Customer'].append(row[6].a.string if row[6].a else row[6].get_text(strip=True))
            launch_dict['Launch outcome'].append(list(row[7].strings)[0])
            launch_dict['Booster landing'].append(landing_status(row[8]))

df = pd.DataFrame({k: pd.Series(v) for k, v in launch_dict.items()})
print('scraped rows:', len(df))
df.head()

scraped rows: 121


,Flight No.,Date,Time,Version Booster,Launch site,Payload,Payload mass,Orbit,Customer,Launch outcome,Booster landing
0,1,4 June 2010,18:45,F9 v1.07B0003.1,CCAFS,Dragon Spacecraft Qualification Unit,0,LEO,SpaceX,Success,Failure
1,2,8 December 2010,15:43,F9 v1.07B0004.1,CCAFS,Dragon,0,LEO,NASA,Success,Failure
2,3,22 May 2012,07:44,F9 v1.07B0005.1,CCAFS,Dragon,525 kg,LEO,NASA,Success,No
3,4,8 October 2012,00:35,F9 v1.07B0006.1,CCAFS,SpaceX CRS-1,"4,700 kg",LEO,NASA,Success,No attempt
4,5,1 March 2013,15:10,F9 v1.07B0007.1,CCAFS,SpaceX CRS-2,"4,877 kg",LEO,NASA,Success,No


In [6]:
df.to_csv('spacex_web_scraped.csv', index=False)
print('saved spacex_web_scraped.csv')

saved spacex_web_scraped.csv
